In [219]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from imblearn.over_sampling import RandomOverSampler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# 1.Binary Classification Naive Bayes 

In [220]:
df = pd.read_csv('../0.dataset/dataset_Kinerja_Karyawan_clean.csv')
df_x = df.drop(columns='Jenis_Kelamin')
df_y = df['Jenis_Kelamin']

In [221]:
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

oversampling= RandomOverSampler(random_state=42)
X_train,y_train = oversampling.fit_resample(X_train,y_train)

In [222]:
params = {
   'var_smoothing': np.logspace(-9, -12, num=100)
}
nby =GaussianNB()
random_search = RandomizedSearchCV(estimator=nby,param_distributions=params,n_iter=20,
                                   cv=6,scoring='accuracy',random_state=42,n_jobs=-1)
random_search.fit(X_train,y_train)
best_model_nby = random_search.best_estimator_

sfs_dct = SequentialFeatureSelector(estimator=best_model_nby,n_features_to_select=23,direction='backward')
sfs_dct.fit(X_train,y_train)

X_train_selected = sfs_dct.transform(X_train)
X_test_selected = sfs_dct.transform(X_test)

fitur_terpilih = X_train.columns[sfs_dct.get_support()]
best_model_nby.fit(X_train_selected,y_train)

print(f'\nHyperparameter Terbaik: {random_search.best_params_}')
print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')


Hyperparameter Terbaik: {'var_smoothing': np.float64(3.0538555088334124e-12)}

Akurasi Validasi Terbaik: 49.47

Fitur Terbaik Yang Terpilih:
['Departemen_Finance', 'Departemen_HR', 'Departemen_IT', 'Departemen_Marketing', 'Departemen_Operations Departemen', 'Status_Pernikahan_Belum Menikah', 'Status_Pernikahan_Cerai', 'Status_Pernikahan_Menikah', 'Pendidikan_Terakhir_D3', 'Pendidikan_Terakhir_S1', 'Pendidikan_Terakhir_S2', 'Pendidikan_Terakhir_S3', 'Pendidikan_Terakhir_SMA/SMK', 'Level_Jabatan_Junior', 'Level_Jabatan_Lead/Manager', 'Level_Jabatan_Mid-Level', 'Level_Jabatan_Senior', 'Rating_Kinerja_Baik', 'Rating_Kinerja_Cukup', 'Rating_Kinerja_Kurang', 'Rating_Kinerja_Sangat Baik', 'Status_Ekonomi_Keatas', 'Status_Ekonomi_Kebawah']


In [223]:
test_accuracy = best_model_nby.score(X_test_selected,y_test)
train_accruacy = best_model_nby.score(X_train_selected,y_train)

y_pred_test = best_model_nby.predict(X_test_selected)
y_prob_test = best_model_nby.predict_proba(X_test_selected)

y_pred_train = best_model_nby.predict(X_train_selected)
y_prob_train = best_model_nby.predict_proba(X_train_selected)

print(f'\nAkurasi Pada Data Test: {test_accuracy*100:.2f}')
print(f'\nAkurasi Pada Data Train: {train_accruacy*100:.2f}')


Akurasi Pada Data Test: 49.40

Akurasi Pada Data Train: 51.60


In [224]:
def evaluate_model(pred,test,prob,evaluate,model_name='Naive Bayes'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test,pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:,1])
    logloss = log_loss(test,prob)

    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }

    df_result = pd.DataFrame(data)
    return df_result

In [225]:
def check_model_fit(df_eval_train,df_eval_test):
    df_combined = pd.concat([df_eval_train,df_eval_test],ignore_index=True)
    accuracy_train = df_eval_train['Accuracy'].values[0]
    accuracy_test = df_eval_test['Accuracy'].values[0]
    gap = accuracy_train - accuracy_test

    if accuracy_train < 0.60 and accuracy_test <0.60:
        status = 'Undeerfitting (Akurasi Rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f})'
    elif gap < -0.05:
        status = 'Overfitting / Data Leakage (Test > Train)'
    else:
        status = 'Good Fit'

    df_combined['Status'] = status
    return df_combined

In [226]:
df_eval_train = evaluate_model(y_pred_train,y_train,y_prob_train,evaluate='Training')
df_eval_test = evaluate_model(y_pred_test,y_test,y_prob_test,evaluate='Test')
df_eval = check_model_fit(df_eval_train,df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
         Model Evaluated  Accuracy  Precision    Recall  F1-Score   ROC-AUC  Log Loss                          Status
0  Naive Bayes  Training  0.516034   0.516761  0.494327  0.505295  0.520453  0.693917  Undeerfitting (Akurasi Rendah)
1  Naive Bayes      Test  0.494000   0.501558  0.475862  0.488372  0.500203  0.697702  Undeerfitting (Akurasi Rendah)


